In [ ]:
# ======================================================
# STEP 2: Import dependencies
# ======================================================
import pandas as pd
import numpy as np
import string
import re
import seaborn as sns
import matplotlib.pyplot as plt

import nltk
# NLTK setup
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# import fasttext
# from gensim.models import FastText, Word2Vec
from torch import nn
import torch
from torch.utils.data import DataLoader, Dataset


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

# Path to your folder in Google Drive
folder_path = "/content/drive/MyDrive/Mini"

# Load all four CSVs
fake_politifact = pd.read_csv(os.path.join(folder_path, "politifact_fake.csv"))
real_politifact = pd.read_csv(os.path.join(folder_path, "politifact_real.csv"))
fake_gossipcop = pd.read_csv(os.path.join(folder_path, "gossipcop_fake.csv"))
real_gossipcop = pd.read_csv(os.path.join(folder_path, "gossipcop_real.csv"))
welfake = pd.read_csv(os.path.join(folder_path, "WELFake_Dataset.csv"))
news = pd.read_csv(os.path.join(folder_path, "news.csv"))


# Add labels: 0 = fake, 1 = real
fake_politifact["label"] = 0
real_politifact["label"] = 1
fake_gossipcop["label"] = 0
real_gossipcop["label"] = 1

# Merge into one DataFrame
df1 = pd.concat([fake_politifact, real_politifact, fake_gossipcop, real_gossipcop])
df2 = pd.concat([welfake, news])
df1["combined_text"] = df1["title"].astype(str)
df1 = df1[["combined_text", "label"]]

df2["combined_text"] = df2["title"].astype(str) + " " + df2["text"].astype(str)
df2 = df2[["combined_text", "label"]]

df = pd.concat([df1, df2])
# Combine the two DataFrames

# Keep only useful columns
fakenewsnet = df1[["combined_text", "label"]]
welfake["combined_text"] = welfake["title"].astype(str) + " " + welfake["text"].astype(str)
welfake = welfake[["combined_text", "label"]]
news["combined_text"] = news["title"].astype(str) + " " + news["text"].astype(str)
news = news[["combined_text", "label"]]
# Save combined dataset
output_path = os.path.join(folder_path, "FakeNewsNet_clean.csv")
df.to_csv(output_path, index=False)

print("✅ Combined dataset saved at:", output_path)
print("📊 Total samples:", len(df))
print(df.columns)
print(fake_politifact.columns)
print(fake_gossipcop.columns)
print(real_politifact.columns)
print(real_gossipcop.columns)
print(welfake.columns)
print(news.columns)
print(fakenewsnet.columns)


In [ ]:
# Show the structure and a few examples
print(df.head())
print("\nDataset Info:")
print(df.info())

# Check for class balance
print("\nLabel Distribution:")
print(df['label'].value_counts())


In [ ]:
# Check missing values in each column
print("\nMissing values per column:")
print(df.isnull().sum())

# Drop rows where title or text is missing
df.dropna(subset=['combined_text', 'label'], inplace=True)
